# Spoken Language Processing


Before you turn this notebook in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).


---

# SLP-Lab1 - Speech Signal Processing 2025-26

_Luis Caldas de Oliveira_

Speech signal processing is a signal processing subfield that deals with the analysis, synthesis, and transformation of speech signals. Speech is a complex signal that conveys information about the content, context, and emotional state of a speaker. The goal of speech signal processing is to extract useful information from speech signals and make it more accessible and useful for various applications.

Speech signal processing techniques play an important role in preparing datasets and in extracting features from speech signals. These features can be used as inputs to machine learning models for various speech-related tasks such as speech recognition, speaker identification, emotion recognition, and language identification.

This lab assignment will introduce some tools and concepts for the manipulation and analysis of speech signals. It includes 20 tasks for the students in the form of short pieces of code that need to be included, metrics that need to be extracted from the speech signal, and answers to questions. Cells with tasks are identified by having Txx in their title, where xx is the order of the task.

This document was inspired by the "Introduction to the acquisition, visualization and processing of speech corpora", "Lab1" and "Lab2" lab assignments by Prof. Isabel Trancoso.

## Introduction

### Group and Student Identification (T01)

Initialize the variable `group_id` with the number that Fenix assigned to your group and `student1_name`, `student1_id`, `student2_name` and `student2_id` with your names and student numbers.


In [ ]:
# YOUR CODE HERE
group_id=7
student1_name="Dinis Alves da Silva"
student1_id=106261
student2_name="Henrique Rodrigues"
student2_id=106362
print(f"Group number: {group_id}")
print(f"Student 1: {student1_name} ({student1_id})")
print(f"Student 2: {student2_name} ({student2_id})")


: 

In [ ]:
assert isinstance(group_id, int) and isinstance(student1_id, int) and isinstance(student2_id, int)
assert isinstance(student1_name, str) and isinstance(student2_name, str) 
assert (group_id > 0) and (group_id < 40)
assert (student1_id > 60000) and (student1_id < 120000) and (student2_id > 60000) and (student2_id < 120000)

### Python Packages

The next step is to define the Python Packages that this notebook requires:
- **NumPy** is a Python library that provides functions to process multidimensional array objects. The NumPy documentation is available here.
- **IPython.display** is a module in the IPython interactive computing environment that provides a set of functions for displaying various types of media in the Jupyter Notebook or other IPython-compatible environments. For example, you can use the `display()` function to display an object in a notebook cell (for example an audio object created with the `Audio()` function).
- **Matplotlib** is a popular Python library that allows users to create a wide range of visualizations using a simple and intuitive syntax.
- **Librosa** is a Python package for analyzing and processing audio signals. It provides a wide range of tools for tasks such as loading and manipulating audio files, extracting features from audio signals, and visualizing and playing back audio data. Librosa is a popular library for use in music information retrieval (MIR) and speech analysis.
- The **SciPy** package provides algorithms for scientific computing in Python.
- The **math** module provides access to the mathematical functions defined by the C standard but that cannot be used with complex numbers

Note: The current Librosa version (0.10.1) requires a Matplotlib version not greater than 3.7.5. You can install it with `pip install matplotlib==3.7.5`.


In [ ]:
import numpy as np
from IPython.display import Audio
from matplotlib import pyplot as plt
import librosa
import librosa.display

import scipy.signal as sig
from scipy.fft import fft
import math

from numpy.testing import assert_array_equal, assert_allclose


## Phonetics

### Harvard sentences

The Harvard sentences are a set of standardized phrases used for speech testing and evaluation of audio equipment, such as telephones, radios, and hearing aids. They were developed by researchers at Harvard University in the early 20th century and have since been widely used as a benchmark for audio quality.

The original set of Harvard sentences consisted of ten lists of ten phrases each, which were carefully designed to include a wide range of speech sounds and phonetic contrasts. Each sentence is relatively short and simple, typically consisting of five to ten words, and is meant to be easy to pronounce and understand.

The Harvard sentences have been revised and expanded over the years, with newer versions containing up to 720 sentences. They are still widely used today in audio testing and research, and have become a standard tool for evaluating speech recognition systems and other audio technologies.

The full list of Harvard sentences is available here: https://www.cs.columbia.edu/~hgs/audio/harvard.html

Philippa Demonte collected a high quality digital audio speech corpus of the Harvard sentences in its entirety (720 phonetically-balanced sentences) recorded December 2018 at the University of Salford with a female native British English speaker. The corpus is avalaible [here](https://salford.figshare.com/articles/media/Speech_corpus_-_Harvard_-_raw_audio/7862666?backTo=/collections/HARVARD_speech_corpus_-_audio_recording_2019/4437578)

The first file includes the following utterances:

0. Harvard list number one.
1. The birch canoe slid on the smooth planks.
2. Glue the sheet to the dark blue background.
3. It's easy to tell the depth of a well.
4. These days a chicken leg is a rare dish.
5. Rice is often served in round bowls.
6. The juice of lemons makes fine punch.
7. The box was thrown beside the parked truck.
8. The hogs were fed chopped corn and garbage.
9. Four hours of steady work faced us.
10. A large size in stockings is hard to sell.



### Audio Recording

The file `harvard01-09.wav` is the recording of the sentence

```
Four hours of steady work faced us.
```

It is recorded in a single channel (mono) at 22050 samples per second with 16 bits per sample and is stored using the wav file format.

Run the next cell to load the recording, plot its time-domain representation, and create a widget to play the file.

In [ ]:
sr = 22050
utt1, sr = librosa.load("harvard01-09.wav", sr=sr)

fig, ax = plt.subplots(figsize=(12,4))
librosa.display.waveshow(utt1, sr=sr, ax=ax)

display(Audio(utt1, rate=sr))

The function `waveshow_seg()` is a helper function to plot a segment of the waveform.

In [ ]:
def waveshow_seg(x, sr, tmin=0, tmax=0):
    if (tmin < 0) or (tmin > x.size/sr):
        tmin = 0
    if (tmax <= 0) or (tmax > x.size/sr): 
        tmax = x.size/sr
    fig, ax = plt.subplots(figsize=(12,4))
    ax.set(xlim=[tmin, tmax])
    librosa.display.waveshow(x, sr=sr, axis='ms')
    return

Similarly, the function `play_seg()` opens a widget to play a segment of the utterance.

In [ ]:
def play_seg(x, sr, tmin=0, tmax=0):
    if (tmin < 0) or (tmin > x.size/sr):
        tmin = 0
    if (tmax <= 0) or (tmax > x.size/sr): 
        tmax = x.size/sr
    n_seg = np.arange(np.floor(tmin*sr), np.ceil(tmax*sr), dtype=int)
    display(Audio(x[n_seg], rate=sr))
    return

In the following cell, you can change the start and end times of the segment of the utterance that you would like to see and hear.

In [ ]:
start = 0.944
end = 1.304
waveshow_seg(utt1, sr, tmin=start, tmax=end)
play_seg(utt1, sr, tmin=start, tmax=end)

### Word Segmentation (T02)

In the previous section, you could notice that the words are not always clearly separated. 

If you have not done so yet, install the free software [Audacity](https://www.audacityteam.org) in your computer. Download the `harvard01-04.wav` file to your computer and open it with Audacity.

Open a label pane and use it to annotate the word boundaries in your file by creating a label for each word (see the Audacity documentation on [label tracks](https://manual.audacityteam.org/man/label_tracks.html)).

Once you have annotated the full utterance with sufficiente precision (< 10 ms), export the labels into a `.txt` file and copy the boundaries of each word into the next code cells.

In [ ]:
word = [\
          [0.252235 ,	0.430538], # four \
          [0.430538 ,	0.876298], # hours \
          [0.876298, 1.037206], # of \
          [1.037206, 1.345976], # steady \
          [1.345976,	1.582989], # work \
          [1.582989,	2.020051], # faced \
          [2.020051,	2.363611], # us \
         ]

# YOUR CODE HERE
wrd = 1
waveshow_seg(utt1, sr, tmin=word[wrd][0], tmax=word[wrd][1])
play_seg(utt1, sr, tmin=word[wrd][0], tmax=word[wrd][1])

In [ ]:
w = 0
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)
    

In [ ]:
w = 1
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)

In [ ]:
w = 2
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)

In [ ]:
w = 3
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)

In [ ]:
w = 4
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)

In [ ]:
w = 5
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)

In [ ]:
w = 6
assert (word[w][0] > 0 and word[w][1] > 0)
assert (word[w][1]-word[w][0]< 0.6)
if (w < len(word) -1):
    assert (abs(word[w+1][0]-word[w][1])<0.0001)

### Phone Segmentation (T03)

Use the tool available at [toPhonetics](https://tophonetics.com) to find the phonetic transcription of the utterance and use Audacity to annotate all the phones of the utterance with an average error of 10 ms.

Once you have annotated the words, export the labels into a `.txt` file and copy the boundaries of each phone into the next code cells.

In [ ]:
phone = [
    [0.252235, 0.298286],  # F
    [0.298286, 0.381934],  # o:
    [0.381934, 0.430545],  # r
    [0.430606, 0.535070],  # a
    [0.535070, 0.651925],  # ʊ
    [0.651925, 0.781080],  # ə
    [0.781080, 0.877946],  # z
    [0.877946, 0.948673],  # ɒ
    [0.948673, 1.037852],  # v
    [1.037852, 1.091666],  # ˈs
    [1.091666, 1.139331],  # t
    [1.139331, 1.199295],  # ɛ
    [1.199295, 1.263873],  # d
    [1.263873, 1.343826],  # i
    [1.343826, 1.406866],  # w
    [1.406866, 1.472981],  # ɜ
    [1.472981, 1.537558],  # ː
    [1.537558, 1.583685],  # k
    [1.583685, 1.659025],  # f
    [1.659025, 1.811243],  # e
    [1.811243, 1.857370],  # ɪ
    [1.857370, 1.929635],  # s
    [1.929635, 2.023426],  # t
    [2.023426, 2.212546],  # ʌ
    [2.212546, 2.364764],  # s
]
ph = 1
waveshow_seg(utt1, sr, tmin=phone[ph][0], tmax=phone[ph][1])
play_seg(utt1, sr, tmin=phone[ph][0], tmax=phone[ph][1])

In [ ]:


assert (abs(phone[0][0]-word[0][0])< 0.01), "The start of the first phone must be aligned the the start of the first word"
assert (abs(phone[len(phone)-1][1]-word[len(word)-1][1])< 0.01), "The end of the last phone must be aligned the the end of the last word"

for p in range(1,len(phone)):
    assert (abs(phone[p][0]-phone[p-1][1])<0.001), "The start of the phone must be close to end of the previous one"

### Fundamental Period (T04)

Estimate the fundamental period at the middle of the first vowel and initialize the variable `vowel1_T0_ms` with its value in miliseconds.

In [ ]:
vowel1_T0_ms = 2.2
# YOUR CODE HERE

print(f"Fundamental period at the middle of the first vowel: {vowel1_T0_ms:.2f} ms")
print(f"Fundamental frequency at the middle of the first vowel: {1000/vowel1_T0_ms:.2f} Hz")

mid = phone[1][0] + (phone[1][1]-phone[1][0])/2

waveshow_seg(utt1, sr, tmin=mid-vowel1_T0_ms/1000, tmax=mid+vowel1_T0_ms/1000)
plt.grid()




In [ ]:
assert (vowel1_T0_ms > 1) and (vowel1_T0_ms < 5)

### Design Your Sentence (T05)

In the next text cell, write a sentence with a minimum of 6 words that includes the names of the students of the group and the vowels [a], [I] or [i], and [u]. The sentence can be in any language that both students speak.
Bellow the sentence write its phonetic transcription in IPA symbols.

Henrique Rodrigues e Dinis Alves grupo sete

### Record Your Sentence (T06)

Use Audacity to record and produce two audio files, one with each students voice saying the sentence that you have designed. Don't forget to trim the recordings but leave around 300 ms of silence at the ends. Use a sample rate of 22050 Hz.

Load the signals to variables `student1_utt` and `student2_utt`.

Place the original wav files in the same folder of this Jupyter Notebook file.

In [ ]:
# YOUR CODE HERE
sr=22050
student1_utt,sr = librosa.load("slphenrique.wav", sr=sr)
student2_utt,sr =librosa.load("slpdinis.wav",sr=sr)

In [ ]:
sr = 22050
fig, ax = plt.subplots(figsize=(12,4))
librosa.display.waveshow(student1_utt, sr=sr, ax=ax)
display(Audio(student1_utt, rate=sr))
fig, ax = plt.subplots(figsize=(12,4))
librosa.display.waveshow(student2_utt, sr=sr, ax=ax)
display(Audio(student2_utt, rate=sr))

## Time-Domain Representations

### Non-Stationary Signals: Signal Framing

Most of real-world signals are non-stationary. Since most signal processing techniques assume that the signal is stationary, it is common to split the original signal into short segments, called frames. To assume that the signal is quasi-stationary in these shorter segments frames are typically chosen to be of 10 to 100 ms in duration. The variable `frame_length` contais the number of samples in each frame.

To prevent large discontinuities when using frames to extract features from the signal, it is also common to use overlapping frames. This means that the next analysis frame includes samples that were also included is previous frames. The variable `hop_length` contais the number of samples between the start of two consecutive frames.

In [ ]:
frame_length = 1024
hop_length = 256

### Zero Padding

In many application we need to reconstruct a signal by recombining overlapping frames. To be able to reconstruct a signal with, **at least**, the same length of the original the signal needs to be extended with zeros. This is called zero padding.



In [ ]:
def pad_signal(x, frame_length, hop_length):
    """Pads the signal x so that it can be split into frames of length frame_length with hop_length in between"""
    N = len(x)
    N_frames = N//hop_length + (N%hop_length > 0)
    x_padded = np.zeros((N_frames-1)*hop_length + frame_length)
    x_padded[:N] = x
    return x_padded



### Chirp Signal

To test the pad_signal() function we will use a we will use a chirp signal. A chirp signal is a type of signal that varies its frequency over time. In this case we will use a linear chirp signal with decreasing frequency.

In [ ]:
dur = 1
c_sr = 22050
t = np.linspace(0, dur, int(dur*c_sr), endpoint=False)
fmax = 300
fmin = 150

f = np.linspace(fmax, fmin, len(t))
c = np.sin(2*np.pi*f*t)
display(Audio(c, rate=sr))

c_pad = pad_signal(c, frame_length, hop_length)
c_nframes = (len(c)+hop_length-frame_length)//hop_length
c_pad_nframes = (len(c_pad)+hop_length-frame_length)//hop_length
print(f"Length of c signal: {len(c)}")
print(f"Number of frames of c signal: {c_nframes}")
print(f"Number of frames of c_pad signal: {c_pad_nframes}")
print(f"Length of reconstructed signal {c_pad_nframes}x{hop_length}={c_pad_nframes*hop_length}")


### Zero Crossing Rate

The number of times a signal crosses the x axis is an indication of the behaviour of the signal. For this reason, the Zero Crossing Rate (ZCR) is a common feature for audio classification.

In [ ]:
def zero_crossing_rate(x):
    if len(x) == 0:
        return 0.0
    return np.sum(np.abs(np.diff(np.signbit(x))))/len(x)

### Frame-Based Zero Crossing Rate (T07)

The previous function computes the zero crossing rate (ZCR) on a segment of the signal. To show how this feature changes in time you need to split the signal in overlapping frames and apply the function zero_crossing_rate() for each frame.

Write a function that computes the ZCR for each frame of lenght `frame_length` with `hop_length` in between.

Use the function `pad_signal()` so that all samples of the input signal are used.

In [ ]:
def frame_based_zcr(x, frame_length, hop_length):
    """Computes the zero crossing rate of signal x by splitting it into frames 
    of length frame_length with hop_length in between."""
    # YOUR CODE HERE
    padded_x=pad_signal(x,frame_length,hop_length)
    zcr=[]
    for start in range(0, len(padded_x) - frame_length + 1,hop_length):
            frame_x=padded_x[start:start+frame_length]
            frame_zero_crossing_rate = zero_crossing_rate(frame_x)
            zcr.append(frame_zero_crossing_rate)
    return np.array(zcr)

In [ ]:
# 1. Type and Shape Checks
x_test_vis = np.array([1, -1, 1, -1, 1, 1, 1, -1])
# Using frame_length=4, hop_length=2
# Padded size should become 10. Number of frames: (10 + 2 - 4)//2 = 4
zcr_vis = frame_based_zcr(x_test_vis, frame_length=4, hop_length=2)

assert isinstance(zcr_vis, np.ndarray), "The output must be a numpy array."
assert zcr_vis.ndim == 1, "The output must be a 1D array."
assert len(zcr_vis) == 4, f"Expected exactly 4 frames for the test signal, but got {len(zcr_vis)}."
print("Visible shape and type tests passed!")


### ZCR of the Chirp Signal

The following code plots the ZCR value for each frame of the previously defined chirp signal.



In [ ]:
plt.plot(frame_based_zcr(c, frame_length, hop_length))
plt.xlabel('Frame number')
plt.ylabel('Zero crossing rate')

### ZCR of Speech 

To understand the meaning of the zero crossing rate feature we will see how it changes over a segment of the utterance `utt1`.

In [ ]:

seg = utt1[int(word[3][0]*sr):int(word[3][1]*sr)]

seg_zcr = frame_based_zcr(seg, frame_length, hop_length)

display(Audio(seg, rate=sr))

t = np.arange(len(seg_zcr)) * hop_length / sr
librosa.display.waveshow(seg, sr=sr)
plt.plot(t, seg_zcr)
plt.show()


### Explain the ZCR Plots (T08)

In the next text cell explain the variation of the ZCR in the previous two plots. How does ZCR change with the fundamental frequency of the signal? For what type of speech sounds has ZCR the largest values? Why is ZCR a relevant feature to characterize the speech signal?

YOUR ANSWER HERE

Chirp signal: The chirp starts at 300 Hz and descends to 150 Hz. Since ZCR tracks how often the waveform crosses zero per sample, it is proportional to the instantaneous frequency. A 300 Hz sine crosses zero 600 times per second; at 22050 Hz sample rate that gives ZCR ≈ 2×300/22050 ≈ 0.027. As f₀ decreases, ZCR decreases proportionally — so the plot shows a falling curve, mirroring the chirp's falling frequency.
Speech signal: In voiced regions (vowels), the vocal folds vibrate at f₀ ≈ 100–250 Hz, producing a relatively low ZCR. In unvoiced regions — especially fricatives like /s/, /ʃ/ — the turbulent noise source excites many high-frequency components that cause the signal to cross zero very rapidly, producing much higher ZCR values. Silent regions produce near-zero ZCR.
Why ZCR is useful: It provides a simple, computationally cheap way to distinguish voiced from unvoiced speech and silence, without needing a full frequency analysis. It is directly related to the dominant frequency of a signal: for a pure sinusoid at frequency ff
f, ZCR =2f/fs= 2f/f_s
=2f/fs​. In the source-filter model (slides SLP5), voiced excitation has low ZCR while noise excitation has high ZCR.

## Frequency-Domain Representations

### Spectrum

The spectrum is a representation of a signal in terms of magnitude and phase characteristics as a function of frequency. The discrete-time Fourier transform (DTFT) can be used to compute the frequency representation of a signal. However, the DTFT is a function of a real variable $\omega \in \mathbb{R}$ that needs to be sampled for use in a digital computer.

The sampled representation of the DTFT is called the discrete Fourier transform (DFT) that can be very efficiently computed using the fast Fourier transform (FFT):
$$
X(k) = \sum_{n=0}^{N-1} x(n) e^{-j \frac{2\pi}{N} kn},\; 0 \leq k \leq N-1
$$

Given the periodicity of the complex exponential $e^{-j \frac{2\pi}{N} kn}$, the definition of $X(k)$ results in a periodic sequence of period $N$. This is prevented by limiting the range of $k$.

The `mag_spectrum()` function plots $|X(k)|$, the magnitude of the spectrum of the sequence $x(n)$.

In [ ]:
def mag_spectrum(x, fs, fmin=0, fmax=0, ax=None):
    """
    Plot the magnitude spectrum of sequence x using the sampling frequency fs
    """

    X = fft(x)
    X_mag = np.absolute(X)
    
    
    # the DFT samples the frequency range with N points
    f = np.linspace(0,fs, X.size)
    
    # plot frequency range
    if (fmin < 0 or fmin>fs):
        fmin = 0
    if (fmax <= 0 or fmax>fs):
        fmax = fs/2
    
    # plot
    if ax == None:
        fig, ax = plt.subplots(figsize=(10,6))
    
    ax.set(xlim=(fmin, fmax))
    ax.plot(f, X_mag)
    plt.xlabel('Frequency (Hz)')
    plt.grid()




### Magnitude Spectrum of a Vowel

The `mag_spectrum()` function can be used to show the magnitude of the power spectrum of the first vowel in the utterance `utt1`.

In [ ]:
vowel = utt1[int(phone[1][0]*sr):int(phone[1][1]*sr)]

mag_spectrum(vowel, sr, fmin=0, fmax=sr)

### Symmetry Property of the DFT

As shown in the previous plot, if $x(n)$ is a sequence of real values, the coefficients of it's DFT have the property that $|X(k)| = |X(N-k)|$. For this reason, we can limit the range of the spectrum to half of the sampling frequency without loss of information.

In [ ]:
mag_spectrum(vowel, sr)

### Decibels relative to full scale (dBFS)

Decibels relative to full scale (dBFS) is a measure of amplitude levels in decibels.

If $v$ is the amplitude of the signal that we want to measure and $v_{0}$ a reference amplitude, the amplitude ratio in decibels is:
$$
L_{dB} = 20 \log_{10}\left( \frac{v}{v_{0}}\right)
$$

The measure of decibels realtive to full scale (dBFS) assumes that $v_0$ is the maximum possible value for $v$ such that:
$$
L_{dBFS}(v_{0}) = 0\ dB
$$

When the amplitude is at 50% of the maximum level:
$$
L_{dBFS}\left( \frac{v_{0}}{2} \right) \approx -6\ dB
$$

Many signals resulting from an analog to digital conversion are represented in an amplitude range of $[-1,1]$, which means that $v_{0}=1$.



In [ ]:
def mag_spectrum_dB(x, fs, fmin=0, fmax=0, ax=None):
    """
    Plot the magnitude spectrum in decibels of sequence x using the sampling frequency fs
    """

    X = fft(x)
    X_mag_dB = 20*np.log10(np.absolute(X))
    
    
    # the DFT samples the frequency range with N points
    f = np.linspace(0,fs, X.size)

    # plot frequency range
    if (fmin < 0 or fmin>fs):
        fmin = 0
    if (fmax <= 0 or fmax>fs):
        fmax = fs/2
    
    # plot
    if ax == None:
        fig, ax = plt.subplots(figsize=(10,6))
    ax.set(xlim=(fmin, fmax))
    ax.plot(f, X_mag_dB)
    plt.xlabel('Frequency (Hz)')
    plt.grid()
    return ax


### Magnitude Spectrum in dBFS


In [ ]:
mag_spectrum_dB(vowel, sr)
plt.show()


### Linear vs Log Spectrum (T09)

In the following text box compare the differences between the two magnitude spectrums plotted earlier. What features are more visible in the first and second plots?

YOUR ANSWER HERE

Linear magnitude spectrum (mag_spectrum): The absolute values ∣X(k)∣|X(k)|
∣X(k)∣ are plotted directly. This makes the strongest harmonics (at multiples of f₀) very prominent and easy to read in amplitude. However, the weaker harmonics at higher frequencies are barely visible because the scale is dominated by the large low-frequency components.
Log-magnitude spectrum in dBFS (mag_spectrum_dB): Amplitudes are plotted as 20log⁡10∣X(k)∣20 \log_{10}|X(k)|
20log10​∣X(k)∣. This compresses the dynamic range and makes the full harmonic series visible, including the weaker high-frequency harmonics. The spectral envelope (the broad shape due to the vocal tract formants — slides SLP5, Multiple Tubes model) becomes easier to identify because the harmonic peaks appear as regular ripples over a smooth formant envelope. The dB scale also matches perceptual loudness more closely.

### Windowing

The function `mag_spectrum_dB()` computes da DFT with the same lenght as the input vector `x`. However, we may want to analyse a fixed-length region of the vowel closer to its middle when the signal is more stable. Also, the FFT is more efficient when the lenght of the signal is a power of 2.

In [ ]:
N = 512
win = np.hanning(N)

midpoint = len(vowel)//2
vowel512 = vowel[midpoint-(N//2):midpoint+(N//2)]*win

librosa.display.waveshow(vowel512, sr=sr)
mag_spectrum_dB(vowel512, sr, fmin=0, fmax=sr/2)

### Narrow-band magnitude spectrum

The DFT samples the DTFT in as many samples as the length of the signal. To have a higher frequency resolution (a narrow-band analysis) we need longer window.

In [ ]:
N = 1024
win = np.hanning(N)

vowel1024 = vowel[midpoint-(N//2):midpoint+(N//2)]*win

librosa.display.waveshow(vowel1024, sr=sr)
mag_spectrum_dB(vowel1024, sr, fmin=0, fmax=sr/2)


### Wide-band magnitude spectrum

With a lower frequency resolution (a wide-band analysis) the envelope of the magnitude spectrum show the resonances of the vocal tract.

In [ ]:
N = 256
win = np.hanning(N)

vowel256 = vowel[midpoint-(N//2):midpoint+(N//2)]*win

librosa.display.waveshow(vowel256, sr=sr)
mag_spectrum_dB(vowel256, sr, fmin=0, fmax=sr/2)



### Harmonic spectrum

The quasi-periodicity of the vowel sound results in the presence of harmonic peaks in the magnitude spectrum.

The frequency representation of the harmonic peaks depend on the window function used to weight the time samples

In [ ]:
N = 1024
win1 = np.ones(N)
win2 = np.hanning(N)

vow_win1 = vowel[midpoint-(N//2):midpoint+(N//2)]*win1
vow_win2 = vowel[midpoint-(N//2):midpoint+(N//2)]*win2

librosa.display.waveshow(vow_win1, sr=sr)
librosa.display.waveshow(vow_win2, sr=sr)
ax = mag_spectrum_dB(vow_win1, sr, fmin=0, fmax=sr/2)
mag_spectrum_dB(vow_win2, sr, fmin=0, fmax=sr/2, ax=ax)


### Spectral Features of Windows Functions (T10) 

In the next text box explain the diferences observed is the magnitude spectrum resulting from the selection of the window. 

YOUR ANSWER HERE

Two windows were compared on a N=1024 segment of the vowel: a rectangular window (win1 = np.ones(N)) and a Hanning window (win2 = np.hanning(N)).
Rectangular window: The abrupt edges at the start and end of the frame act as sharp discontinuities. In the frequency domain this introduces spectral leakage — energy from each harmonic "leaks" into adjacent frequency bins. The sidelobes of the rectangular window's Fourier transform are large (only −13 dB below the main lobe), making it harder to resolve nearby harmonics and causing the spectral floor to be higher.
Hanning window: The smooth tapering to zero at both ends greatly reduces spectral leakage. The sidelobes fall at −31 dB, so harmonics are much cleaner and the spectral floor is lower. The trade-off is a slightly wider main lobe, meaning marginally lower frequency resolution. For voiced speech analysis, the Hanning window is preferred because it reveals the harmonic structure and formant envelope more clearly — as shown in the SLP5 slides on the source-filter model.

### Mel frequency spectrum

The mel spectrum is a frequency representation where the frequencies are spaced acording to the mel scale and not evenly spaced like in the DFT. The mel is scales of frequencies such that each unit is judged by listeners to be equal in pitch distance from the next.

The mel frequency scale can be approximated by:
$$
m(f) = 2595 \log_{10} \left( 1 + \frac{f}{700} \right)
$$

To compute a mel frequency spectrum, a series of overlapping triangle filters are applied to the magnitudes of the FFT spectrum. Each FFT value is multiplied by its corresponding value in each triangle filter. The resulting values are then summed up for each filter, creating a series of energy values in each mel frequency band. This process is often referred to as "mel filtering".

The amplitudes of the filters can be normalized, so that each triangle has the same area, or not normalized, where all the filters have the same amplitude.



In [ ]:
def mel_spectrum_dB(x, fs, n_mels, fmin=0, fmax=0, ax=None):
    """
    Plot the magnitude spectrum of sequence x
    using the sampling frequency fs
    """

    
    # plot frequency range
    if (fmin < 0 or fmin>fs):
        fmin = 0
    if (fmax <= 0 or fmax>fs):
        fmax = fs/2

    # mel scale spectrum  with n_mels bins, norm='slaney' does area normalization
    X = librosa.feature.melspectrogram(y=x, sr=sr, norm='slaney', n_fft=len(x), n_mels=n_mels, center=False)
    X_mag_dB = 10*np.log10(np.abs(X))
    
    # mel scale frequencies
    f = librosa.mel_frequencies(n_mels=n_mels, fmax=sr//2)

    # plot
    if ax == None:
        fig, ax = plt.subplots(figsize=(10,6))
    ax.set(xlim=(fmin, fmax))
    ax.plot(f, X_mag_dB)
    plt.xlabel('Frequency (Hz)')
    plt.grid()
    return ax

### Mel frequency resolution

To represent a signal in a mel scale, one needs to define the number of filters in which to divide the frequency range.

The number of filters used can vary depending on the specific application and the characteristics of the signal being analyzed. Typically, between 20 and 80 filters are used, with higher numbers of filters providing more detailed information about the frequency content of the signal. 

In [ ]:
n_bins = 80

ax = mag_spectrum_dB(vowel1024, sr)
mel_spectrum_dB(vowel1024, sr, n_bins, ax=ax)
plt.show()

### Mel-frequency cepstral coefficients (MFCC)


The computation of the MFCCs takes the following steps:

1. Compute the discrete Fourier transform (DFT) of the signal
$$
X(k)=\sum_{n=0}^{N-1} x(n) e^{-j\frac{2\pi}{N}kn},\;0 \leq k \leq N-1
$$

2. Compute the mel-frequency spectrum:

$$
M(r) = \frac{1}{A_{r}} \sum_{k=L_{r}}^{U_{r}} |V_{r}(k) X(k)|
$$
where $V_{r}(k)$ is the triangular weighting function for the 𝑟-th filter, $L_{r}$ and $U_{r}$ are the lower and upper indices of the frequencies of the triangular filter, $A_r$ is the amplitude normalization factor of the filter. This groups the DFT values in critical bands.

3. Compute the discrete cosine transform (DCT) of the logarithm of the magnitude of the filter outputs:

$$
MFCC(m) = \frac{1}{R} \sum_{r=1}^{R} \ln(M(r)) \cos\left( \frac{\pi}{R}\left( r + \frac{1}{2} \right)m \right)
$$
where $R$ is the number of mel filters.

The MFCC coefficients provide a more compact representation of the signal spectral contents when compared, for example, with the mel-frequency spectrum.

In [ ]:
def mfcc_dB(x, n_mfcc):
    """
    Plot the mel frequency coefficients (MFCC) of sequence x
    """


    # mel scale spectrum  with n_mels bins, norm='slaney' does area normalization
    n_mels = n_mfcc
    X = librosa.feature.melspectrogram(y=x, sr=sr, norm='slaney', n_fft=len(x), n_mels=n_mels, center=False)
    mfccs = librosa.feature.mfcc(S=librosa.power_to_db(X), n_mfcc=n_mfcc)

    k = np.arange(0, n_mfcc, 1)

    # plot

    fig, ax = plt.subplots(figsize=(10,6))
    ax.plot(k, mfccs)
    plt.xlabel('MFCC')
    plt.grid()
    return ax

### MFCC visualization

In [ ]:

n_mfcc = 20

mfcc_dB(vowel1024, n_mfcc)
plt.show()

### short-time Fourier transform (STFT)

The short-time Fourier transform is a type of Fourier analysis used to determine the frequency content of a signal over short, fixed-length time intervals. It is used in many applications, such as speech processing and musical analysis. The STFT is based on the conventional Fourier transform, but it divides the signal into overlapping segments and then performs a Fourier analysis for each segment. This results in a two-dimensional representation of the signal, where the frequency is on one axis and time on the other. Different resolutions can be obtained for analyzing different aspects of the signal by varying the size and position of the segments.

The signal segmentation is performed by multiplying the signal by a window function $w(n)$ that is zero-valued outside a specified interval. For example the rectangular window:
$$
w_{r}(n) =
\begin{cases}
1, & 0 \le n \le M-1 \\
0, & \text{otherwise}
\end{cases}
$$

The short-time Fourier transform, $X(n, \omega)$ is a two-dimensional representation of the signal $x(n)$:
$$
X(n, \omega) = \sum_{m=-\infty}^{+\infty} x(n+m)w(m) e^{-j\omega (n+m)} 
$$

If we use the discrete Fourier transform (DFT):

$$
X(n,k) = \sum_{m=0}^{M-1}x(n+m)w(m) e^{-j \frac{2\pi}{M}(n+m)},\ 0\le k \le M-1
$$


### Spectrogram of a signal

The spectrogram of a time-domain signal is a representation of the magnitude of the short-time Fourier transform (STFT) ($X(n,k)$) of a signal.

To facilitate the analysis of the signal, it is common to use the time axis in seconds or milliseconds and the frequency axis in Hz. The value of the amplitude of the spectrum at each point is represented with a darker color for low values and brighter color for higher values.

For better visualization of the entire range of amplitudes, it is frequent to represent them in decibels (dB).

The short-time Fourier transform can be computed with the `librosa.stft()` function. The resulting linear amplitudes need to be converted to dB. The function `librosa.display.specshow()` function can then be used to display the spectrogram as a frequency versus time representation of the signal.

In [ ]:
def spectrogram(x, fs, n_fft=512, win_length=512, hop_length=64, window='hann'):
    D = librosa.stft(x, n_fft=n_fft, win_length=win_length, hop_length=hop_length, window=window)
    DAbsdB = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    return DAbsdB



### Spectrogram of a Speech Segment

Use the spectrogram() function to plot the spectrogram of a segment of `utt1`

In [ ]:

DAbsdB = spectrogram(seg, sr)
fig, ax = plt.subplots(figsize=(10,8))
img = librosa.display.specshow(DAbsdB, ax=ax, x_axis='time', y_axis='linear')
ax.set(title='Linear-frequency power spectrogram')
ax.label_outer()
fig.colorbar(img, ax=ax, format="%+2.f dB")

Audio(data=seg, rate=sr)

### Waveform and Spectrogram

It is frequently useful to have a simultaneous view of both the spectrogram and the time domain waveform aligned in time.

In [ ]:
def spec_wave_show(x, fs, tmin=0, tmax=0, n_fft=1024, win_length=1024, hop_length=128, window='hann'):
    """
    Plot a spectrogram and a waveform of a signal x
    """
    if (tmax == 0):
        tmax = x.shape[0]/fs
    nseg = np.arange(np.floor(tmin*fs), np.floor(tmax*fs), dtype=int)

    D = librosa.stft(x, n_fft=n_fft, win_length=win_length, hop_length=hop_length, window=window)
    DAbsdB = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    fig, ax = plt.subplots(figsize=(10,8), nrows=2, sharex=True, gridspec_kw={'height_ratios': [4, 1]})
    ax[0].set(xlim=[tmin, tmax])
    librosa.display.specshow(DAbsdB, n_fft=n_fft, win_length=win_length, hop_length=hop_length, sr=fs, ax=ax[0], x_axis='time', y_axis='linear')
    ax[0].set(title='Linear-frequency power spectrogram')
    librosa.display.waveshow(x, sr=fs, axis='time')
    display(Audio(data=x[nseg], rate=fs))
    plt.tight_layout()
    plt.show()
    return fig, ax

img, ax = spec_wave_show(seg, sr)

### Visualization of Plosives (T11)

In the previous spectrogram the plosives are not very visible in the spectrogram. Selected a different frame size to make it more noticeable by changing the value of `fsize`.


In [ ]:
fsize = 1024
# YOUR CODE HERE
raise NotImplementedError()
img, ax = spec_wave_show(seg, sr, n_fft=fsize, win_length=fsize, hop_length=fsize//4, window='hann')

In [ ]:
assert (fsize & (fsize-1) == 0) and fsize != 0

### Mel Spectrogram

All frequency analysis made for a segment of the vowel I can be represented in a spectrogram plot. Such is the case of the mel-frequency spectrum

In [ ]:
def mel_spectrogram(y, sr, n_mels=80, n_fft=1024, hop_length=256):
    # Compute mel-spectrogram
    mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, power=1.0, n_fft=n_fft, hop_length=hop_length, norm='slaney')
    mel_spect_dB = librosa.amplitude_to_db(mel_spect, ref=np.max)
    return mel_spect_dB

### Example of a Mel Spectrogram

In [ ]:
mel_spect = mel_spectrogram(seg, sr, n_mels=80)
plt.figure(figsize=(12, 4))
librosa.display.specshow(mel_spect, sr=sr, hop_length=256, y_axis='mel', x_axis='time')
plt.title('Mel-Spectrogram')
plt.colorbar(format='%+2.0f dB')
plt.show()

### MFCC Spectrogram

The mel-frequency cepstral coefficients (MFCC) can also be represented in a spectrogram.

In [ ]:
def mfcc(y, sr, n_mels=64, n_mfcc=13):
    # Compute mel-spectrogram
    mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, power=1.0, n_fft=1024, hop_length=256, norm='slaney')
    mel_spect_dB = librosa.amplitude_to_db(mel_spect, ref=np.max)
    # Compute MFCCs from mel-spectrogram
    mfccs = librosa.feature.mfcc(S=mel_spect_dB, n_mfcc=n_mfcc)
    return mfccs

### Example of a MFCC Spectrogram

The MFCCs can capture the spectral stability. They are also less sentive to variations of the fundamental frequency than other representations

In [ ]:
mfccs = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=20)
fig, ax = plt.subplots(figsize=(10,8), nrows=3, sharex=True, gridspec_kw={'height_ratios': [3, 3, 1]})
librosa.display.specshow(mel_spect, hop_length=256, ax=ax[0], x_axis='time', y_axis='mel')
librosa.display.specshow(mfccs, ax=ax[1], x_axis='time')
ax[1].set(ylabel='MFCC')
librosa.display.waveshow(seg, sr=sr, axis='time', ax=ax[2])
plt.tight_layout()

### Fundamental Frequency Estimation and Voice/Unvoice Detection

The fundamental frequency is the frequency of the first harmonic of a periodic signal. In speech processing the fundamental frequency is often referred by f0 or pitch.

In this lab we will use the estimator pyin() provide by the libROSA package.

To show the performance of the algorithm in tracking the first harmonic of the signal we can plot the spectrogram in a logarithmic frequency scale and overlap the f0 values produced by the estimator.

You can see the f0 decay typical of a declarative utterance. You also observe the intonation breaks when the decaying pattern is interrupted.


In [ ]:
f0, voiced_flag, voiced_probs = librosa.pyin(utt1, fmin=100, fmax=360)
times = librosa.times_like(f0)
D = librosa.amplitude_to_db(np.abs(librosa.stft(utt1)), ref=np.max)
fig, ax = plt.subplots(figsize=(10,8), nrows=2, sharex=True, gridspec_kw={'height_ratios': [4, 1]})
img = librosa.display.specshow(D, x_axis='time', y_axis='log', ax=ax[0])
ax[0].set(title='pYIN fundamental frequency estimation')
#fig.colorbar(img, ax=ax, format="%+2.f dB")
ax[0].plot(times, f0, label='f0', color='cyan', linewidth=4)
ax[0].legend(loc='upper right')
librosa.display.waveshow(utt1, sr=sr, axis='time', ax=ax[1])
plt.tight_layout()
display(Audio(data=utt1, rate=sr))

### Interpolation of F0 Values

The frame-based fundamental frequency estimator produces a f0 value for every frame. We want to interpolate these values to have a value of f0 for every sample of the signal.

The function scipy.interpolate.interp1d() can help with the interpolation.

In [ ]:
from scipy.interpolate import interp1d

def sig_f0(x, sr, frame_length, hop_length):
    """Estimates and interpolates the F0 for every time sample of given signal x
    with sample-accurate time alignment."""

    N = len(x)
    x_pad = pad_signal(x, frame_length, hop_length)
    
    f0, voiced_flag, voiced_probs = librosa.pyin(x_pad, fmin=60, fmax=360, sr=sr, 
                                                 frame_length=frame_length, 
                                                 hop_length=hop_length,
                                                 center=False)
    
    # 1. Sample-accurate time alignment
    t_f0 = np.arange(len(f0)) * hop_length + (frame_length / 2.0)
    t = np.arange(N)
    
    # 2. Create interpolator (using extrapolate to handle the edge gaps)
    # Note: f0 contains NaNs for unvoiced regions. interp1d safely propagates 
    # these NaNs to the interpolated samples, which is exactly what we want!
    interpf0 = interp1d(t_f0, f0, kind='linear', bounds_error=False, fill_value="extrapolate")
    
    # 3. Interpolate over target samples
    f0i = interpf0(t)
    
    # 4. Memory Efficiency: Replace NaNs with 0 in-place
    np.nan_to_num(f0i, copy=False, nan=0.0)

    return f0i

### Visualization of the Interpolated F0

In [ ]:
fsize = 1024
hop = 256
utt1_f0 = sig_f0(utt1, sr, fsize, hop)
t = np.linspace(0, len(utt1)/sr, len(utt1))
fig, ax = plt.subplots(nrows=2, sharex=True)
ax[0].plot(t, utt1_f0, label='f0')
librosa.display.waveshow(utt1, sr=sr, axis='time', ax=ax[1])
plt.tight_layout()

### Interpolated RMS Values (T12)


The root mean square (RMS) of a sequence of $N$ values is the square root of the arithmetic mean of the squares of the values, that is
$$
x_{RMS}=\sqrt{ \frac{1}{N} \sum_{n=0}^{N-1}\left[x(n)\right]^{2}}
$$

The value $(x_{RMS})^{2}$ is the _power_ of a signal, that is, its _energy_ over a period time.

Write a function that computes the RMS value for each frame, anchoring each value to the middle sample of its respective frame to ensure sample-accurate time alignment. To create a more natural-sounding envelope, interpolate these values in the log domain using `np.log()` producing a final envelope vector equal in length to the original speech signal. You may want to use the `pad_signal()` and `interp1d()` functions.


In [ ]:
from scipy.interpolate import interp1d

def sig_rms(x, frame_length, hop_length):
    """Computes and interpolates RMS values for every sample of a given signal x 
    with sample-accurate time alignment."""
# YOUR CODE HERE
raise NotImplementedError()
    return rmsi[0:N]

In [ ]:

import numpy as np
from scipy.interpolate import interp1d
from numpy.testing import assert_array_equal, assert_allclose

# ==========================================
# VISIBLE TESTS (Sanity checks for students)
# ==========================================
x_test_vis = np.ones(400) # Simple constant signal
rms_vis = sig_rms(x_test_vis, frame_length=100, hop_length=50)

assert isinstance(rms_vis, np.ndarray), "The output must be a numpy array."
assert rms_vis.ndim == 1, "The output must be a 1D array."
assert len(rms_vis) == len(x_test_vis), f"Output length ({len(rms_vis)}) must match input length ({len(x_test_vis)})."

print("Visible shape and length tests passed!")

# ==========================================
# HIDDEN TESTS (Grading checks)
# ==========================================


### Visualization of the Interpolated RMS

In [ ]:
fsize = 1024
hop = 256
t = np.linspace(0, len(utt1)/sr, len(utt1))
utt1_rms = sig_rms(utt1, fsize, hop)
fig, ax = plt.subplots(nrows=2, sharex=True)
ax[0].plot(t, utt1_rms, label='RMS')
librosa.display.waveshow(utt1, sr=sr, axis='time', ax=ax[1])
plt.tight_layout()

## Model of Speech Production

### Formants

The resonances of the vocal tract, known as formants, can be accurately approximated as second order resonant filters. These formants are created by the shaping of the vocal tract, which acts as a series of resonant cavities. Each of these cavities has a resonant frequency, which determines the frequency at which it will naturally vibrate.

By using second order resonant filters to model these formants, the formant synthesizer can simulate the acoustic characteristics of the vocal tract, allowing it to produce vowel sounds and other vocal sounds.

In this example we will model the vocal tract with a cascade of two second order resonant filters.

### Second-Order Discrete-Time System


A second-order discrete-time system has the following transfer function
$$
H(z) = \frac{K}{1-a_1 z^{-1} - a_2 z^{-2}}
$$
where $K$ is the static gain.

The transfer function that can be written in terms of its poles:
$$
H(z) = K \frac{(1-z_1 z^{-1})(1-z_2 z^{-1})}{(1-p_1 z^{-1})(1-p_2 z^{-1})}
$$

where $p_1$ and $p_2$ are the complex roots of the denominator of $H(z)$ known as poles.

If $K=1$, the system can be implemented as
$$
y(n) = x(n) + a_1 y(n-1) + a_2 y(n-2)
$$
where
$$
\begin{align}
a_1 &= p_1 + p_2\\
a_2 &= - p_1 p_2
\end{align}
$$

For this system to work as a resonator the poles must be complex conjugates of each other, that is, $p_1 = p_2^{\ast}$. In this case:
$$
\begin{align}
a_1 &= 2\Re{p_1}\\
a_2 &= - |p_1|^2
\end{align}
$$
A resonator is characterized by its natural frequency $\omega_n$ and damping ratio $zeta$. When $0<\zeta<1$ the system is underdamped and the poles are complex conjugates. 

When modeling the vocal tract each formant is defined by a frequency and a bandwidth,

The formant frequency is usually considered the undamped resonant frequency (or natural frequency):
$$
F = 2 \pi \omega_n
$$

The bandwidth is measured between the cutoff frequencies, most frequently defined as the frequencies at which the frequency response has fallen to half the value at its peak.
$$
B = f_{c_2} - f_{c_1} = \frac{\omega_n \zeta}{\pi}
$$

These continuous-time frequency values can be used to define the position of the poles of the continuous time transfer function:
$$
\omega_{peak} = \omega_n \sqrt{1-2\zeta^2}
$$

To find the corresponding discrete-time transfer function we need to map the poles into the z-plane:
$$
\begin{align}
p_1 &= e^{s_1 T}=e^{(-\zeta \omega_n + \omega_n \sqrt{\zeta^2 -1})T}\\
p_2 &= e^{s_2 T}=e^{(-\zeta \omega_n - \omega_n \sqrt{\zeta^2 -1})T}\\
\end{align}
$$



In [ ]:

def formant_filter_coeffs(F, B, sr):
    T = 1/sr
    wn = 2*np.pi*F
    zeta = 2*np.pi*B/(2*wn)
    s1 = -zeta*wn + 1j*wn*np.sqrt(1-zeta**2)
    s2 = -zeta*wn - 1j*wn*np.sqrt(1-zeta**2)
    p1 = np.exp(s1*T)
    p2 = np.exp(s2*T)
    a = np.poly([p1, p2])
    b = np.array([np.sum(a)])
    return b, a


### Formant Filter Frequency Response

The function scypi.signal.freqz() can be used to get the samples of the frequency response of a system with a rational transfer function such as the formant filter.

In [ ]:
fsyn_sr = 8000
F1 = 500
B1 = 100

b, a = formant_filter_coeffs(F1, B1, fsyn_sr)
w, h = sig.freqz(b, a, worN=1024)
f = w/(2*np.pi)*fsyn_sr
plt.plot(f, 20*np.log10(abs(h)))
plt.grid()

### Cascate of Two Formant Filters

Connecting the output of one formant filter to another results in the product of the transfer function of both filters
$$
H(z) = \frac{1}{1-a_{11} z^{-1} - a_{12} z^{-2}} \frac{1}{1-a_{11} z^{-1} - a_{22} z^{-2}}
$$
The resulting filter coefficients are the convolution of the filter parameters
$$
H(z) = \frac{1}{1 - 
(a_{11}+a_{12}) z^{-1} - 
(a_{12}-a_{11}a_{21}+a_{22}) z^{-2} -
(-a_{11}a_{22}-a_{12}a_{21}) z^{-3} -
(-a_{12}a_{22})z^{-4}}
$$



In [ ]:

F2 = 1500
B2 = 100
b1, a1 = formant_filter_coeffs(F1, B1, fsyn_sr)
b2, a2 = formant_filter_coeffs(F2, B2, fsyn_sr)
b = np.convolve(b1, b2)
a = np.convolve(a1, a2)
w, h = sig.freqz(b, a, worN=1024)
f = w/(2*np.pi)*fsyn_sr
plt.plot(f, 20*np.log10(abs(h)))
plt.grid()


### Source Signal Generator (T13)

For voiced sounds ($f_0>0$) the source signal is a train of impulses:
$$
e(n) = \sqrt{P} \sum_{k=-\infty}^{+\infty} \delta(n-kP)
$$
where $P=1/f_0$ is the fundamental period, and $\delta(n)$ is the unit impulse function.

For unvoiced sounds ($f_0=0$) the source signal is zero-mean unit-variance Gaussian white noise:
$$
e(n) \sim \mathcal{N} (0,1)
$$

Write a function that takes a f0 vector generated by the function sig_f0() and returns a vector with the same lenght with pulses or noise depending on the f0 estimate. Use basic Python and NumPy functions.


In [ ]:
def source_generator_old(f0, sr):
    """Generates a periodic pulse train with given fundamental frequency or with random noise if f0=0"""
# YOUR CODE HERE
raise NotImplementedError()
    return exc

def source_generator(f0, sr):
    """Generates a periodic pulse train with given fundamental frequency or with random noise if f0=0"""
# YOUR CODE HERE
raise NotImplementedError()
    return exc


In [ ]:

sr_test = 16000
f0_test = np.ones(100) * 200
exc_vis = source_generator(f0_test, sr_test)

# 1. Type and Shape Checks
assert isinstance(exc_vis, np.ndarray), "The output must be a numpy array."
assert exc_vis.ndim == 1, "The output must be a 1D array."
assert len(exc_vis) == len(f0_test), f"Output length ({len(exc_vis)}) must match input length ({len(f0_test)})."
assert not np.isnan(exc_vis).any(), "Your output contains NaNs. Check for division by zero!"

# 2. Simple Behavioral Checks
assert np.any(exc_vis > 0), "Excitation should contain pulses (values > 0) when f0 > 0."

exc_unv_vis = source_generator(np.zeros(100), sr_test)
assert np.any(exc_unv_vis != 0), "Unvoiced sections (f0=0) should contain noise, not just zeros."

print("Visible tests passed!")


### Synthesize 3 Vowels (T14)

In the following code block generate 3 audio signals:
- `vowel_a` with samples of a synthesized vowel [a]
- `vowel_i` with samples of a synthesized vowel [I]
- `vowel_u` with samples of a synthesized vowel [u]

with the following characteristics

- Sample rate: 16000 Hz
- Use only two formants for each vowel
- Duration: 0.5 s
- Linearly decaying f0 from 220 Hz to 190 Hz
- Raised cosine RMS: starts and ends in zero and maximum amplitude at the middle of the vowel.

The values for the formant frequencies and bandwidths should be extracted from your recordings


In [ ]:
fsyn_sr = 16000

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
display(Audio(vowel_a, rate=fsyn_sr))
display(Audio(vowel_i, rate=fsyn_sr))
display(Audio(vowel_u, rate=fsyn_sr))

### Linear Prediction Analysis

Linear prediction tries to predict a signal sample $\hat{s}(n)$ using a linear combination of the signal's past samples:
$$
\hat{s}(n) = \sum_{k=1}^{P} a_k s(n-k)
$$

The prediction error $e(n)$ is called the residue:
$$
e(n) = s(n) - \sum_{k=1}^{P} a_k s(n-k)
$$

This difference equation as the form of an all-zero filter with a transfer function $A(z)$ that is a polynomial in $z$:
$$
A(z) = 1 - \sum_{k=1}^{P} a_k z^{-k}
$$

The inverse of this filter is an all-pole filter that can be seen as similar to the vocal tract transfer function. In this case the residue can also be seen as an estimate of the glottal excitation.

If the signal is assumed to be quasi-stationary, the linear prediction coefficients $a_k$ can be found by finding the values that minimizes the energy of the prediction error (residue):
$$
a_k = \text{argmin}  \sum_{n=-\infty}^{+\infty} \left( e(n) \right)^2
$$

This is an optimization problem where we want to minimize the error function:
$$
\cal{E} = \sum_{n=-\infty}^{+\infty} \left( s(n) - \sum_{k=1}^{P} a_k s(n-k) \right)^2
$$

Setting the derivative of the error to zero
$$
\frac{d \cal{E}}{d a_k} = 2 \sum_{n=-\infty}^{+\infty} \left( s(n) - \sum_{k=1}^{P} a_k s(n-k) \right) s(n-k) = 0
$$

Results in a set of $P$ equations with P unknowns ($a_k$). Using the autocorrelation function:
$$
R(k) = \sum_{n=-\infty}^{+\infty} s(n) s(n-k)
$$

The system of equations becomes:
$$
\mathbf{R} \mathbf{a} = \mathbf{\gamma}
$$
where $\mathbf{R}$ is a $P \times P$ matrix with elements $R_{k,i}=R(|k-i|)$, $\mathbf{a}$ is $P \times 1$ vector with the lpc coefficients $a_k$ and $\mathbf{\gamma}$ is $P \times 1$ vector with $\gamma_k = R(k)$.

The solution is:
$$
\mathbf{a} = R^{-1} \mathbf{\gamma}
$$

All the elements along the diagonals of the $\mathbf{R}$ matrix have the same value, that is, $\mathbf{R}$ is a Toeplitz matrix. The system of equations can be solved using the Levinson-Durbin recursion.

In [ ]:
def bias_autocorr(signal, order):
    """Computes the autocorrelation of signal up to the order of the LPC analysis"""
    N = len(signal)
    r = np.zeros(order+1)
    r[0] = np.sum(signal**2)/N
    for k in range(1,order+1):
        r[k] = np.sum(signal[k:]*signal[:-k])/N
    return r

In [ ]:
def levison_durbin(r, order):
    """Solves the levinson-durbin recursion for the LPC analysis"""
    g = r[1] / r[0]
    a = np.array([g])
    v = (1 - g**2) * r[0]
    for m in range(1, order):
        g = (r[m+1] - np.dot(a, r[1:m+1])) / v
        a = np.r_[ g, a - g*a[m-1::-1] ]
        v *= 1 - g**2
    return np.r_[1, -a[::-1]]


In [ ]:
def lpc(signal, order):
    """Computes the LPC analysis of signal x up to order order"""
    return levison_durbin(bias_autocorr(signal, order), order)

### LPC Spectrum

The LPC spectrum is the frequency response of the all-pole system with the coefficients resulting from LPC analysis. The transfer function of the system is:
$$
H(z) = \frac{K}{A(z)}
$$
where $K$ is a gain factor.

To find the frequency response $z=e^{j\omega}$:
$$
H(e^{j\omega}) = \frac{K}{A(e^{j\omega})}
$$

The inverse filter is an all-zero filter, that is, a finite impulse response filter:
$$
H^{-1}(e^{j\omega}) = A(e^{j\omega})
$$
with impulse response:
$$
h^{-1}(n) = 1 - \sum_{k=1}^{P} a_k \delta(n-k)
$$
where $\delta(n)$ is the unit impulse.

The frequency response can be computed by inverting the DFT of the impulse response of the inverse filter.

In [ ]:
def lpc_spectrum_dB(x, fs, order, fmin=0, fmax=0, ax=None):
    """
    Plot the magnitude spectrum of sequence x
    using the sampling frequency fs
    """

    X = np.divide(1.0, fft(lpc(x, order), len(x)))
    X_mag_dB = 20*np.log10(np.abs(X))
    
    # the DFT samples the frequency range with N points
    f = np.linspace(0,fs, X.size)
    
    # plot frequency range
    if (fmin < 0 or fmin>fs):
        fmin = 0
    if (fmax <= 0 or fmax>fs):
        fmax = fs/2
    
    # plot
    if ax == None:
        fig, ax = plt.subplots(figsize=(10,6))
    ax.set(xlim=(fmin, fmax))
    ax.plot(f, X_mag_dB)
    plt.xlabel('Frequency (Hz)')
    plt.grid()
    return ax

### LPC Order

The order of the LPC analysis should be adjusted in order to model the resonances of the vocal tract in the selected frequency range.

The analysis of a lossless tube show that it has one resonance in every 1000 Hz frequency band. Since each ressonance requires 2 filter coefficients, it is common to chose an LPC order equal to sampling frequency divide by 1000 plus an addicional ressonance to allow the presence of a nasal zero.

In [ ]:
order = sr//1000 + 2

ax = mag_spectrum_dB(vowel1024, sr)
ax = lpc_spectrum_dB(vowel1024, sr, order, ax=ax)
plt.show()

### LPC Error (T15)

The all-zeros filter can be used to estimate the LPC error or residue. The residue signal should have less energy than the original signal and shows periodic pulses.

If we assume that the LPC filter is an estimate of the vocal tract, residue signal is an estimate of the voicing source necessary to produce the vowel.

The function scipy.signal.lfilter() implements a difference equation. Initialize the vectors `b` and `a` with the filter parameters

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()
vowel1024_resid = sig.lfilter(b, a, vowel1024)
librosa.display.waveshow(vowel1024, sr=sr)
librosa.display.waveshow(vowel1024_resid, sr=sr)
plt.grid()


In [ ]:
assert np.average(np.abs(vowel1024_resid)) < np.average(np.abs(vowel1024))

## SpeechT5 Hifi-GAN Neural Vocoder

In the previous exercise, we used Linear Predictive Coding (LPC) to estimate the spectral envelope of a vowel. By treating the vocal tract as a series of filters, we could visualize the resonant frequencies (formants) that define different speech sounds.

However, generating high-quality speech from these parameters using classical methods is difficult and often sounds robotic. To achieve human-like quality, we use a Neural Vocoder like the one found in the SpeechT5 framework.

SpeechT5 is a unified, multimodal Transformer model, developed and open-sourced by Microsoft Research Asia, that can handle various speech tasks—such as text-to-speech, speech-to-text, and speech enhancement—by mapping both text and speech into a shared hidden space using pre-trained encoders and decoders.

For text-to-speech, it utilizes a two-stage pipeline. In the first stage, SpeechT5 converts text into a Log-Mel Spectrogram. The second stage takes that spectrogram and converts it into a speech signal; the module that performs this second process is often referred to as a vocoder (from 'voice coder').

The SpeechT5 vocoder is based on HiFi-GAN (High-Fidelity Generative Adversarial Network). This architecture uses a 'Generator' to create audio and 'Discriminators' to ensure that the audio sounds like a real human voice rather than a robotic approximation. It is incredibly fast, often generating audio 100x+ faster than real-time on a GPU.

The pre-trained weights for the SpeechT5 HiFi-GAN vocoder are publicly available on Hugging Face under the MIT License, allowing for free use in both academic and commercial applications.






### Step 1: Load the Neural Vocoder (HiFi-GAN)

To utilize the SpeechT5 HiFi-GAN, we must first load its pre-trained weights from Hugging Face. Unlike classical DSP functions that rely on fixed mathematical formulas, a neural vocoder consists of millions of tunable parameters that have been "trained" on thousands of hours of human speech.

In [ ]:
from transformers import SpeechT5HifiGan

print("Loading pre-trained HiFi-GAN vocoder...")
# We use a vocoder trained by Microsoft, readily available on Hugging Face
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

### Step 2: Resample speech signal

Most neural speech models, including SpeechT5 and its HiFi-GAN vocoder, are pre-trained on large-scale datasets like LibriTTS, which are standardized at a 16,000 Hz sampling rate. Because the model’s internal filters and learned patterns are tuned to this specific resolution, feeding it audio at a different rate (like 44.1 kHz or 48 kHz) would cause a "mismatch" in the frequency representation, resulting in distorted or unintelligible audio. By resampling our signals to 16 kHz, we ensure our data aligns perfectly with the model's expectations, maintaining the correct pitch and temporal characteristics during processing.

In [ ]:
new_sr = 16000
utt1_rs = librosa.resample(utt1, orig_sr=sr, target_sr=new_sr)
student1_rs = librosa.resample(student1_utt, orig_sr=sr, target_sr=new_sr)
student2_rs = librosa.resample(student2_utt, orig_sr=sr, target_sr=new_sr)

### Step 3: Extract the Acoustic Features (Mel-Spectrogram)

Neural vocoders are highly sensitive to the specific mathematical configuration of their input. While a standard spectrogram represents frequency linearly, a Log-Mel Spectrogram transforms the data to match human biology: the Mel scale warps frequencies to match how we perceive pitch, and the Log transformation scales amplitudes to match how we perceive loudness.

To test the vocoder, we will reconstruct the speech signal from its Log-Mel Spectrogram. For the SpeechT5 HiFi-GAN to reconstruct the audio correctly, we must precisely match the digital signal processing (DSP) parameters used during its training. This includes normalizing the audio to a specific peak amplitude, removing any DC offset, and utilizing "Slaney" normalization for the Mel filters. Even small deviations in these settings—such as using the wrong FFT size or a different log-base—can lead to severe artifacts or a complete loss of speech intelligibility in the output.



#### Function to compute the Log Mel Spectrogram for SpeechT5

To communicate with the SpeechT5 HiFi-GAN, we must format our audio into a Log-Mel Spectrogram. We have provided a helper function speecht5_log_mel_spectrogram(y) below. This function handles the complex normalization and warping required by the model.

Key Parameters to note:
- `target_sr = 16000` it only works at this sampling rate
- `n_mels=80`: The model 'sees' 80 frequency bands.
- `hop_length=256`: The model processes a new 'slice' of audio every 16ms (256 samples at 16kHz).
- `log10`: The amplitudes are converted to a logarithmic scale to match human perception.

In [ ]:

def speecht5_log_mel_spectrogram(y):
    # Fixed sampling rate required by SpeechT5
    target_sr = 16000 

    # 1. Normalization: Peak-normalize and remove DC offset
    y = 0.9 * y / np.max(np.abs(y))
    y = y - np.mean(y)

    # 2. Compute Mel Spectrogram
    mel_spectrogram = librosa.feature.melspectrogram(
        y=y, 
        sr=target_sr,
        n_fft=1024, 
        hop_length=256, 
        n_mels=80, 
        fmin=0, 
        fmax=8000, 
        power=1.0,
        htk=False, 
        norm='slaney'
    )

    # 3. Dynamic Range Clipping: Prevents log(0) and limits noise floor
    mel_spectrogram = np.clip(mel_spectrogram, a_min=1e-5, a_max=None)

    # 4. Log Transformation
    return np.log10(mel_spectrogram)

#### Spectral Ratio (T16)

Given a start and end time in secondons, write a function that uses `speecht5_log_mel_spectrogram()` to compute the Spectral Ratio (the difference between high and low energy in the log domain) for a segment on an audio signal and test it of the first and second phones of `utt1`.

In [ ]:
def compute_segment_spectral_ratio(y, start_sec, end_sec):
    """
    Computes the ratio of high-frequency energy (>4kHz) to 
    low-frequency energy (<4kHz) for a specific time window.
    """
    sr = 16000
    hop_length = 256
    
# YOUR CODE HERE
raise NotImplementedError()
    return spectral_ratio

In [ ]:
# Calculate ratios for the first two phones
p1_start, p1_end = phone[0][0], phone[0][1]
p2_start, p2_end = phone[1][0], phone[1][1]

ratio_p1 = compute_segment_spectral_ratio(utt1_rs, p1_start, p1_end)
ratio_p2 = compute_segment_spectral_ratio(utt1_rs, p2_start, p2_end)

print(f"Phone 1 Spectral Ratio: {ratio_p1:.4f}")
print(f"Phone 2 Spectral Ratio: {ratio_p2:.4f}")

assert np.isscalar(ratio_p1), "Function must return a single scalar."

# Fricatives should have much higher ratios than vowels
if ratio_p1 > ratio_p2:
    print(f"✅ Success: first phone has a higher spectral ratio than the second.")
else:
    print(f"ℹ️ Note: second phone has a higher spectral ratio than the first. Check if this matches the phonetics!")


### Step 4: Synthesize Audio using the Neural Vocoder

With the Log-Mel Spectrogram prepared, we can now pass it to the neural vocoder to reconstruct the speech signal. This step involves shifting the data from NumPy into a PyTorch Tensor.




#### Function to Synthesize Audio

Neural networks are specifically architected to process data in "batches" to maximize efficiency. Even though we are only synthesizing one utterance at a time, we must "fake" a batch by adding an extra dimension (index 0) to our tensor. Furthermore, we must transpose the matrix: while librosa structures data as (bins, frames) for spectral analysis, the SpeechT5 Transformer expects (frames, bins), treating the time sequence as the primary axis.

In [ ]:
import torch


def synthesize_audio(log_mel, vocoder):
    """
    Synthesizes a waveform from a log-mel spectrogram using a neural vocoder.
    
    Parameters:
    log_mel (np.ndarray): The log-mel spectrogram of shape (num_mel_bins, num_frames).
    vocoder: The pre-trained SpeechT5HifiGan model.
    
    Returns:
    np.ndarray: The 1D synthesized audio signal.
    """

    # 1. Transpose and Tensor Conversion
    # Librosa: (80, frames) -> PyTorch: (frames, 80)
    # .unsqueeze(0) adds the batch dimension: (1, frames, 80)
    spectrogram_tensor = torch.tensor(log_mel).T.unsqueeze(0).float()

    # 2. Neural Inference
    # We use torch.no_grad() because we are not training the model;
    # this reduces memory usage and speeds up the process.
    with torch.no_grad():
        waveform = vocoder(spectrogram_tensor)

    # 3. Post-processing
    # squeeze() removes the batch and channel dimensions
    # cpu().numpy() ensures the data is back in a standard NumPy format
    synthesized_audio = waveform.squeeze().cpu().numpy()

    return synthesized_audio

#### Resinthesize 3 utterances (T17)

Resynthesize the three resampled utterances using your functions. Store the results in variables named: `utt1_synth`, `student1_synth`, `student2_synth`

In [ ]:


# YOUR CODE HERE
raise NotImplementedError()

display(Audio(utt1_synth, rate=new_sr))
display(Audio(student1_synth, rate=new_sr))
display(Audio(student2_synth, rate=new_sr))

### Temporal Manipulation: Time-Stretching

One of the advantages of using a neural vocoder is that the "how" (the spectral envelope) is separated from the "when" (the temporal axis). If we stretch or compress the Log-Mel Spectrogram along the time axis, we change the speed of the speech without significantly altering the pitch or the identity of the speaker. This is much more effective than simply speeding up a waveform, which would result in the "chipmunk effect."

By applying a stretch factor (s), we transform a spectrogram of length T into a new matrix of length T⋅s. We can achieve this through linear interpolation, effectively "drawing out" the vowels or "squeezing" the consonants.

#### Time-Stretch Function (T18)

Create a function that stretches or compresses the spectrogram along the time axis by a positive real factor.

In [ ]:
from scipy.ndimage import zoom

def stretch_log_mel(log_mel, factor):
    """
    Stretches or compresses the spectrogram along the time axis.
    
    Parameters:
    log_mel (np.ndarray): The log-mel spectrogram (num_mel_bins, num_frames)
    factor (float): The stretch factor. 
                    > 1.0 makes speech slower. 
                    < 1.0 makes speech faster.
    
    Returns:
    np.ndarray: The manipulated spectrogram.
    """
# YOUR CODE HERE
raise NotImplementedError()
    return stretched_mel

In [ ]:
# Create a test case
original_mel = speecht5_log_mel_spectrogram(utt1_rs)
num_mels, original_frames = original_mel.shape

# Test: Slowing down by 1.5x
slow_factor = 1.5
slow_mel = stretch_log_mel(original_mel, slow_factor)

print(f"Original shape: {original_mel.shape}")
print(f"Stretched shape: {slow_mel.shape}")

slow_audio = synthesize_audio(slow_mel, vocoder)


# --- Assertions ---
assert slow_mel.shape[0] == num_mels, "Error: You should not change the number of Mel bins!"
assert np.isclose(slow_mel.shape[1], int(original_frames * slow_factor), atol=1), \
    f"Expected roughly {int(original_frames * slow_factor)} frames, but got {slow_mel.shape[1]}."

print("✅ Shape test passed!")



#### Time-strectch test

To test the time-stretching function we will will generate two new versions for each of our three speakers:
- A fast version using a stretch factor of 0.6.
- A slow version using a stretch factor of 1.4.

In [ ]:

def process_stretch(y, factor, vocoder):
    mel = speecht5_log_mel_spectrogram(y)
    stretched_mel = stretch_log_mel(mel, factor)
    return synthesize_audio(stretched_mel, vocoder)

# Utterance 1
utt1_fast = process_stretch(utt1_rs, 0.6, vocoder)
utt1_slow = process_stretch(utt1_rs, 1.4, vocoder)
display(Audio(utt1_rs, rate=16000))
display(Audio(utt1_fast, rate=16000))
display(Audio(utt1_slow, rate=16000))

# Student 1
student1_fast = process_stretch(student1_rs, 0.6, vocoder)
student1_slow = process_stretch(student1_rs, 1.4, vocoder)
display(Audio(student1_rs, rate=16000))
display(Audio(student1_fast, rate=16000))
display(Audio(student1_slow, rate=16000))

# Student 2
student2_fast = process_stretch(student2_rs, 0.6, vocoder)
student2_slow = process_stretch(student2_rs, 1.4, vocoder)
display(Audio(student2_rs, rate=16000))
display(Audio(student2_fast, rate=16000))
display(Audio(student2_slow, rate=16000))

### Spectral Shifting: Frequency Manipulation

This final modification is where we truly "hack" the speaker's identity. In the previous step, we manipulated time. Now, we will manipulate the Spectral Envelope by shifting the frequency bins of the Log-Mel Spectrogram.

By shifting the energy in the Log-Mel Spectrogram vertically, we are effectively moving the resonant frequencies (formants) and harmonics of the voice.
- Shifting Up: Shifting the bins to higher frequencies makes the vocal tract sound smaller (like a child or a smaller person).
- Shifting Down: Shifting the bins to lower frequencies makes the vocal tract sound larger and deeper.

Unlike a simple pitch shift in the time domain, shifting bins in a Mel Spectrogram changes the spectral color of the voice. Because the Mel scale is quasi-logarithmic, a linear shift of bins


#### Frequency Shift Function (T19)

Write a function that shifts the mel bins by constant positive or negative integer amount.

In [ ]:
def shift_log_mel(log_mel, shift_bins):
    """
    Shifts the mel bins up or down by a constant integer amount.
    
    Parameters:
    log_mel (np.ndarray): The log-mel spectrogram (80, num_frames)
    shift_bins (int): Number of bins to shift. 
                      Positive moves energy to higher frequencies.
                      Negative moves energy to lower frequencies.
    
    Returns:
    np.ndarray: The frequency-shifted spectrogram with same shape as input.
    """
    num_mels, num_frames = log_mel.shape
    # Initialize the output with our "noise floor" (-5.0) 
    # to avoid artifacts in the empty space.
    shifted_mel = np.full_like(log_mel, -5.0)
    
# YOUR CODE HERE
raise NotImplementedError()
    return shifted_mel

In [ ]:
# Create a test case
original_mel = speecht5_log_mel_spectrogram(utt1_rs)
shift_val = 10 # Shift up significantly

shifted_up = shift_log_mel(original_mel, shift_val)

# --- Test 1: Shape Consistency ---
assert shifted_up.shape == original_mel.shape, (
    f"Shape mismatch! Expected {original_mel.shape}, got {shifted_up.shape}. "
    "Ensure you are not changing the number of Mel bins."
)

# --- Test 2: Directional Check ---
# When shifting UP, the mean of the top 10 bins should increase 
# compared to the original, as energy is moved into them.
orig_high_mean = np.mean(original_mel[-10:, :])
shift_high_mean = np.mean(shifted_up[-10:, :])

print(f"Original high-freq mean: {orig_high_mean:.4f}")
print(f"Shifted high-freq mean: {shift_high_mean:.4f}")

if shift_high_mean > orig_high_mean:
    print("✅ Logic Check: Energy successfully moved toward higher bins.")
else:
    print("❌ Logic Check: Energy did not move up. Check your slicing indices!")


#### Frequency Shift Test

To test the frequency-shifting function we will will generate two new versions for each of our three speakers:
- A high version using a shift o +5.
- A low version using a shift o -5.

In [ ]:

def process_shift(y, shift, vocoder):
    mel = speecht5_log_mel_spectrogram(y)
    shifted_mel = shift_log_mel(mel, shift)
    return synthesize_audio(shifted_mel, vocoder)

# Utterance 1
utt1_high = process_shift(utt1_rs, +5, vocoder)
utt1_low = process_shift(utt1_rs, -5, vocoder)
display(Audio(utt1_rs, rate=16000))
display(Audio(utt1_high, rate=16000))
display(Audio(utt1_low, rate=16000))

# Student 1
student1_high = process_shift(student1_rs, +5, vocoder)
student1_low = process_shift(student1_rs, -5, vocoder)
display(Audio(student1_rs, rate=16000))
display(Audio(student1_high, rate=16000))
display(Audio(student1_low, rate=16000))

# Student 2
student2_high = process_shift(student2_rs, +5, vocoder)
student2_low = process_shift(student2_rs, -5, vocoder)
display(Audio(student2_rs, rate=16000))
display(Audio(student2_high, rate=16000))
display(Audio(student2_low, rate=16000))